# 🔋 EVolvAI — Physics-Informed Generative Counterfactual VAE
## Academic SOTA Edition v2.0 · Climate Fusion & Optimized Capacity

This notebook contains the complete, reproducible training pipeline for the EVolvAI research paper.

### Methodological Upgrades in v2.0:
1. **PST Climate Fusion:** Fetches Open-Meteo data and shifts UTC→PST to align with Caltech EV charging behaviour.
2. **Local `weather_data.parquet` Overlay:** If `weather_data.parquet` exists in the project root (Acedonia, CA data), it overrides the Open-Meteo fetch for matching dates.
3. **Capacity Optimization:** TCN reduced to ~4M params to prevent KLD-induced posterior collapse.
4. **Synthetic Solar Modeling:** Approximates winter irradiance curves.
5. **Comprehensive Diagnostics:** Loss curve, R² curve, confusion matrix, MAE, zero-output check, latent PCA.

## 1 · Environment Setup & Drive Mount

In [ ]:
import sys
import os

# ── Google Drive integration (safe: works on Colab and local alike) ──────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/EVolvAI_Research'
    print("🌐 Running on Google Colab — Drive mounted.")
except ImportError:
    # Local environment: use a local directory that mirrors Drive structure
    DRIVE_DIR = os.path.join(os.getcwd(), 'google_drive_sync', 'EVolvAI_Research')
    print(f"💻 Running locally — checkpoints will go to: {DRIVE_DIR}")

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
print(f"\n💾 Checkpoint dir: {DRIVE_DIR}")

# ── Install / Import ─────────────────────────────────────────────────────────
import subprocess, sys as _sys
def _ensure(*pkgs):
    for p in pkgs:
        try:
            __import__(p.split('[')[0].replace('-','_'))
        except ImportError:
            subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', p])
_ensure('torch', 'numpy', 'pandas', 'pyarrow', 'scipy', 'requests',
        'scikit-learn', 'matplotlib')

import json
import torch
import requests
import datetime
import pandas as pd
import numpy as np
import urllib.parse
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')          # headless-safe; swap to 'inline' for Jupyter inline
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, confusion_matrix, mean_absolute_error
from sklearn.decomposition import PCA

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🐍 Python : {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"⚙️  Device : {DEVICE}")

## 2 · Unified Data Extraction (ACN + Localized Open-Meteo + Local Weather Overlay)

In [ ]:
# --- A. ACN-DATA API EXTRACTION (Target: Jan-Feb 2019) ---
API_TOKEN = '-RQtRLh5YnSZMVsanUCuLAa1DGvtdX35Q2qt5Dv5FA8'
HEADERS = {'Authorization': f'Bearer {API_TOKEN}'}
SITE = 'caltech'

print("📥 Fetching EV charging logs from Caltech ACN...")
query = {"$and": [{"connectionTime": {"$gte": "Tue, 01 Jan 2019 00:00:00 GMT"}},
                  {"connectionTime": {"$lte": "Thu, 28 Feb 2019 23:59:59 GMT"}}]}
url_acn = f"https://ev.caltech.edu/api/v1/sessions/{SITE}?where={urllib.parse.quote(json.dumps(query))}"

try:
    res_acn = requests.get(url_acn, headers=HEADERS, timeout=30)
    res_acn.raise_for_status()
    sessions = res_acn.json().get('_items', [])
    records = []
    for s in sessions:
        if not s.get('connectionTime') or not s.get('kWhDelivered'): continue
        conn_time = pd.to_datetime(s['connectionTime']).tz_convert('America/Los_Angeles')
        records.append({
            'date': conn_time.date().isoformat(),
            'hour': conn_time.hour,
            'node_id': hash(s.get('spaceID', 'unknown')) % 32,
            'demand_kw': s['kWhDelivered'] / max((s.get('duration', 1) / 60), 1)
        })
    df_ev = pd.DataFrame(records, columns=['date', 'hour', 'node_id', 'demand_kw'])
    df_ev.to_parquet('data/processed/train_data.parquet')
    print(f"   ✓ Fetched & localized {len(df_ev)} charging sessions.")
except Exception as e:
    print(f"   ⚠️  ACN API Fetch Failed: {e}")

# --- B. OPEN-METEO WEATHER EXTRACTION & TIMEZONE FIX ---
print("\n📥 Fetching historical climate data for Pasadena, CA (Open-Meteo)...")
url_meteo = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 34.1478, "longitude": -118.1445,
    "start_date": "2019-01-01", "end_date": "2019-02-28",
    "hourly": ["temperature_2m", "precipitation"],
    "timezone": "UTC"
}

try:
    res_meteo = requests.get(url_meteo, params=params, timeout=30)
    res_meteo.raise_for_status()
    w_data = res_meteo.json()
    df_w = pd.DataFrame({
        "datetime_utc": pd.to_datetime(w_data["hourly"]["time"]),
        "temperature_c": w_data["hourly"]["temperature_2m"],
        "precipitation_mm": w_data["hourly"]["precipitation"]
    })
    df_w['datetime_local'] = df_w['datetime_utc'].dt.tz_localize('UTC').dt.tz_convert('America/Los_Angeles')
    df_w['date'] = df_w['datetime_local'].dt.date.astype(str)
    df_w['hour'] = df_w['datetime_local'].dt.hour
    def approx_solar(h):
        return max(0.0, np.cos((h - 12) * np.pi / 10)) if 7 <= h <= 17 else 0.0
    df_w["solar_availability"] = df_w['hour'].apply(approx_solar)
    df_w["traffic_index"] = 0.5
    df_w = df_w[['date', 'hour', 'temperature_c', 'precipitation_mm', 'solar_availability', 'traffic_index']]
    df_w.to_parquet('data/processed/weather_data.parquet')
    print(f"   ✓ Fetched & time-shifted {len(df_w)} hours of climate data.")
except Exception as e:
    print(f"   ⚠️  Weather API Failed: {e}")


# --- C. LOCAL weather_data.parquet OVERLAY (Acedonia, CA) ---
# This overlays real observed weather for matching dates (no data removed).
LOCAL_WEATHER_PATH = os.path.join(os.getcwd(), 'weather_data.parquet')
if os.path.exists(LOCAL_WEATHER_PATH):
    print(f"\n🌤️  Found local weather_data.parquet — overlaying Acedonia, CA data...")
    try:
        df_local_w = pd.read_parquet(LOCAL_WEATHER_PATH)
        feat_cols = ['temperature_c', 'precipitation_mm', 'solar_availability', 'traffic_index']
        if all(c in df_local_w.columns for c in ['date', 'hour'] + feat_cols):
            df_local_w['date'] = df_local_w['date'].astype(str)
            if os.path.exists('data/processed/weather_data.parquet'):
                df_w_base = pd.read_parquet('data/processed/weather_data.parquet')
                # Overlay: update rows where local data matches date+hour
                df_w_base = df_w_base.set_index(['date', 'hour'])
                df_local_w = df_local_w.set_index(['date', 'hour'])
                df_w_base.update(df_local_w[feat_cols])
                df_w_base = df_w_base.reset_index()
                df_w_base.to_parquet('data/processed/weather_data.parquet')
                print(f"   ✓ Overlaid {len(df_local_w)} local weather rows onto processed weather.")
            else:
                df_local_w.reset_index().to_parquet('data/processed/weather_data.parquet')
                print(f"   ✓ Saved local weather as primary weather source.")
        else:
            print(f"   ⚠️  Local parquet missing required columns — skipping overlay.")
    except Exception as e:
        print(f"   ⚠️  Local weather overlay failed: {e}")
else:
    print("\nℹ️  No local weather_data.parquet found in project root — using Open-Meteo data only.")

## 3 · Hyperparameters & SOTA Configuration
*Parameters reduced from 28M to ~4M to cure the negative $R^2$ variance collapse.*

In [ ]:
class CFG:
    NUM_NODES = 32
    SEQ_LEN = 24
    NUM_WEATHER_FEATURES = 4
    NUM_FEATURES = NUM_NODES + NUM_WEATHER_FEATURES

    # Stage 1: Physics zeroed (pre-training)
    LAMBDA_VOLT    = 0.0
    LAMBDA_THERMAL = 0.0
    LAMBDA_XFMR   = 0.0

    EPOCHS        = 150
    BATCH_SIZE    = 64
    LEARNING_RATE = 2e-4
    GRAD_CLIP_NORM = 1.0

    # OPTIMIZED CAPACITY (~4M Params)
    TCN_CHANNELS   = [128, 256, 256, 256]
    KERNEL_SIZE    = 3
    DROPOUT        = 0.15
    LATENT_DIM     = 128
    COND_DIM       = 6
    DECODER_HIDDEN = 512

    LOG_EVERY      = 5       # print every N epochs
    SAVE_EVERY     = 10      # checkpoint every N epochs

    CHECKPOINT_FILE = os.path.join(DRIVE_DIR, 'gcvae_sota_v2_checkpoint.pt')

print(f"CFG: {CFG.NUM_FEATURES} input features | {CFG.LATENT_DIM}-D latent | {CFG.EPOCHS} epochs")

## 4 · Data Augmentation Engine

In [ ]:
class EVDemandDataset(Dataset):
    def __init__(self):
        print("\n⚙️ Fusing ACN Demand Data with Localized Climate Data...")

        df_ev = pd.read_parquet("data/processed/train_data.parquet")
        df_w  = pd.read_parquet("data/processed/weather_data.parquet")

        pivot_ev = df_ev.pivot_table(index=["date", "hour"], columns="node_id",
                                      values="demand_kw", fill_value=0.0)
        days, conditions, weather_features = [], [], []
        monthly_mean_temp = df_w['temperature_c'].mean()

        for date in pivot_ev.index.get_level_values('date').unique():
            day_demand  = pivot_ev.loc[date].values
            day_weather = df_w[df_w['date'] == date].sort_values('hour')

            if len(day_demand) == CFG.SEQ_LEN and len(day_weather) == CFG.SEQ_LEN:
                # A. Spatial Augmentation
                if day_demand.shape[1] < CFG.NUM_NODES:
                    needed = CFG.NUM_NODES - day_demand.shape[1]
                    clones = day_demand[:, np.random.choice(day_demand.shape[1], needed, replace=True)]
                    noise  = np.random.normal(0, 0.02 * clones.std() + 1e-4, clones.shape)
                    day_demand = np.concatenate([day_demand, np.clip(clones + noise, 0, None)], axis=-1)

                days.append(day_demand)
                weather_features.append(
                    day_weather[['temperature_c','precipitation_mm','solar_availability','traffic_index']].values
                )

                # B. Causal Condition Vector
                dt_obj = pd.to_datetime(date)
                temp_anomaly = (day_weather['temperature_c'].mean() - monthly_mean_temp) / \
                               (day_weather['temperature_c'].std() + 1e-5)
                conditions.append([
                    float(temp_anomaly),
                    1.0,
                    float(day_weather['solar_availability'].mean()),
                    1.0 if dt_obj.weekday() >= 5 else 0.0,
                    0.0,
                    float(day_weather['traffic_index'].mean())
                ])

        raw_demand     = np.stack(days)
        raw_conditions = np.stack(conditions).astype(np.float32)
        raw_weather    = np.stack(weather_features)

        # C. Temporal Augmentation (Scale & Shift)
        aug_demand = raw_demand.copy()
        scales     = np.random.uniform(0.85, 1.15, (aug_demand.shape[0], 1, CFG.NUM_NODES))
        aug_demand = aug_demand * scales
        for i, shift in enumerate(np.random.choice([-1, 0, 1], size=aug_demand.shape[0])):
            aug_demand[i] = np.roll(aug_demand[i], shift, axis=0)

        demand_full  = np.concatenate([raw_demand, aug_demand], axis=0)
        weather_full = np.concatenate([raw_weather, raw_weather], axis=0)
        self.conditions = np.concatenate([raw_conditions, raw_conditions], axis=0)

        # D. Z-Score Normalization
        demand_norm = (demand_full - demand_full.mean()) / (demand_full.std() + 1e-8)
        self.data   = np.concatenate([demand_norm, weather_full], axis=-1).astype(np.float32)

        # Store raw demand for confusion matrix evaluation
        self.raw_demand = demand_full.astype(np.float32)

        print(f"   ✓ SOTA Dataset Ready: {self.data.shape} | Conditions: {self.conditions.shape}")

    def __len__(self):         return len(self.data)
    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx]).permute(1, 0), torch.from_numpy(self.conditions[idx])

## 5 · Lean Attention-Augmented Model

In [ ]:
class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1, groups=1, bias=True):
        self._causal_padding = (kernel_size - 1) * dilation
        super().__init__(in_channels, out_channels, kernel_size, stride=stride,
                         padding=self._causal_padding, dilation=dilation, groups=groups, bias=bias)
    def forward(self, x):
        out = super().forward(x)
        return out[:, :, :-self._causal_padding] if self._causal_padding != 0 else out

class TCNBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, dropout):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(n_inputs,  n_outputs, kernel_size, stride=stride, dilation=dilation),
            nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(n_outputs, n_outputs, kernel_size, stride=stride, dilation=dilation),
            nn.ReLU(), nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(self.net(x) + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size, dropout):
        super().__init__()
        layers = []
        for i, out_ch in enumerate(num_channels):
            in_ch = num_inputs if i == 0 else num_channels[i - 1]
            layers.append(TCNBlock(in_ch, out_ch, kernel_size, stride=1, dilation=2**i, dropout=dropout))
        self.network = nn.Sequential(*layers)
    def forward(self, x): return self.network(x)

class GenerativeCounterfactualVAE(nn.Module):
    def __init__(self):
        super().__init__()
        tcn_flat = CFG.SEQ_LEN * CFG.TCN_CHANNELS[-1]
        self.encoder_tcn = TemporalConvNet(CFG.NUM_FEATURES, CFG.TCN_CHANNELS, CFG.KERNEL_SIZE, CFG.DROPOUT)
        self.attention   = nn.MultiheadAttention(embed_dim=CFG.TCN_CHANNELS[-1], num_heads=4, batch_first=True)
        self.fc_mu       = nn.Linear(tcn_flat, CFG.LATENT_DIM)
        self.fc_logvar   = nn.Linear(tcn_flat, CFG.LATENT_DIM)

        self.decoder_fc  = nn.Sequential(
            nn.Linear(CFG.LATENT_DIM + CFG.COND_DIM, CFG.DECODER_HIDDEN), nn.ReLU(),
            nn.Linear(CFG.DECODER_HIDDEN, tcn_flat), nn.ReLU()
        )
        self.decoder_tcn = TemporalConvNet(CFG.TCN_CHANNELS[-1], [64, CFG.NUM_FEATURES], CFG.KERNEL_SIZE, CFG.DROPOUT)

    def encode(self, x):
        h = self.encoder_tcn(x)
        h_permuted = h.permute(0, 2, 1)
        attn_out, _ = self.attention(h_permuted, h_permuted, h_permuted)
        h_flat = attn_out.flatten(start_dim=1)
        return self.fc_mu(h_flat), self.fc_logvar(h_flat)

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def decode(self, z, condition):
        h = self.decoder_fc(torch.cat([z, condition], dim=-1)).view(-1, CFG.TCN_CHANNELS[-1], CFG.SEQ_LEN)
        return self.decoder_tcn(h)

    def forward(self, x, condition):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar), condition), mu, logvar


def vae_loss_function(recon_x, x, mu, logvar, kld_weight: float = 0.0):
    recon = F.mse_loss(recon_x, x, reduction='mean')
    kld   = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + (kld_weight * kld), recon.item(), kld.item()


total_params = sum(p.numel() for p in GenerativeCounterfactualVAE().parameters())
print(f"🧠 Model parameters: {total_params:,}  (~{total_params/1e6:.1f}M)")

## 6 · Resilient Training Engine (with Full Metric Tracking)

In [ ]:
def train_model():
    model     = GenerativeCounterfactualVAE().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE)
    dataset   = EVDemandDataset()
    loader    = DataLoader(dataset, batch_size=CFG.BATCH_SIZE, shuffle=True)

    print(f"\n🚀 Training on: {DEVICE}")
    print(f"   Dataset: {len(dataset)} samples | {len(loader)} batches/epoch")

    # ── History buffers ──────────────────────────────────────────────────────
    history = {
        'epoch':      [],
        'total_loss': [],
        'recon_loss': [],
        'kld_loss':   [],
        'r2':         [],
        'mae':        [],
        'kld_weight': [],
    }

    # ── Checkpoint resume ─────────────────────────────────────────────────────
    start_epoch = 1
    if os.path.exists(CFG.CHECKPOINT_FILE):
        ck = torch.load(CFG.CHECKPOINT_FILE, map_location=DEVICE)
        model.load_state_dict(ck['model_state_dict'])
        optimizer.load_state_dict(ck['optimizer_state_dict'])
        start_epoch = ck['epoch'] + 1
        if 'history' in ck:
            history = ck['history']
        print(f"🔄 Resuming from Epoch {start_epoch}")

    # ── Training loop ─────────────────────────────────────────────────────────
    model.train()
    for epoch in range(start_epoch, CFG.EPOCHS + 1):
        epoch_total = epoch_recon = epoch_kld = 0.0
        all_targets, all_preds = [], []

        # Cyclic KLD annealing capped at 0.05
        current_kld = min(0.05, (epoch - 1) / 100.0)

        for batch_x, batch_cond in loader:
            x, cond = batch_x.to(DEVICE), batch_cond.to(DEVICE)
            optimizer.zero_grad()
            recon, mu, logvar = model(x, cond)
            loss, r, k = vae_loss_function(recon, x, mu, logvar, kld_weight=current_kld)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP_NORM)
            optimizer.step()

            epoch_total += loss.item()
            epoch_recon += r
            epoch_kld   += k
            all_targets.append(x.detach().cpu().numpy())
            all_preds.append(recon.detach().cpu().numpy())

        # ── Epoch metrics
        n_batch = max(len(loader), 1)
        avg_total = epoch_total / n_batch
        avg_recon = epoch_recon / n_batch
        avg_kld   = epoch_kld   / n_batch

        t_flat = np.concatenate(all_targets).flatten()
        p_flat = np.concatenate(all_preds).flatten()
        r2     = r2_score(t_flat, p_flat)
        mae    = mean_absolute_error(t_flat, p_flat)

        history['epoch'].append(epoch)
        history['total_loss'].append(avg_total)
        history['recon_loss'].append(avg_recon)
        history['kld_loss'].append(avg_kld)
        history['r2'].append(r2)
        history['mae'].append(mae)
        history['kld_weight'].append(current_kld)

        if epoch % CFG.LOG_EVERY == 0 or epoch == 1:
            print(f"Epoch {epoch:>3}/{CFG.EPOCHS} | Loss: {avg_total:.5f} "
                  f"(Recon: {avg_recon:.5f}, KLD: {avg_kld:.5f}) | "
                  f"R²: {r2:+.4f} | MAE: {mae:.4f} | KLD-w: {current_kld:.3f}")

        if epoch % CFG.SAVE_EVERY == 0 or epoch == CFG.EPOCHS:
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss':                 avg_total,
                'history':              history,
            }, CFG.CHECKPOINT_FILE)
            print(f"   💾 Checkpoint saved (epoch {epoch})")

    print("\n✅ Training complete!")
    return model, history, dataset


trained_model, history, dataset = train_model()

## 7 · Loss & R² Curves

In [ ]:
epochs = history['epoch']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('EVolvAI Training Diagnostics', fontsize=15, fontweight='bold')

# ── Total loss ────────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(epochs, history['total_loss'], color='#e74c3c', linewidth=2, label='Total Loss')
ax.set_title('Total VAE Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# ── Recon vs KLD ─────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(epochs, history['recon_loss'], color='#3498db', linewidth=2, label='Reconstruction (MSE)')
ax2 = ax.twinx()
ax2.plot(epochs, history['kld_loss'], color='#9b59b6', linewidth=2, linestyle='--', label='KLD')
ax.set_title('Reconstruction vs KLD'); ax.set_xlabel('Epoch')
ax.set_ylabel('Recon MSE', color='#3498db')
ax2.set_ylabel('KLD', color='#9b59b6')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

# ── R² over epochs ───────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(epochs, history['r2'], color='#27ae60', linewidth=2)
ax.axhline(0, color='red', linewidth=1, linestyle='--', label='R²=0 (baseline)')
best_r2 = max(history['r2'])
best_ep = epochs[history['r2'].index(best_r2)]
ax.scatter([best_ep], [best_r2], color='#f39c12', zorder=5,
           label=f'Best R²={best_r2:.4f} @ E{best_ep}')
ax.set_title('R² (Variance Explained)'); ax.set_xlabel('Epoch'); ax.set_ylabel('R²')
ax.legend(); ax.grid(True, alpha=0.3)

# ── MAE over epochs ──────────────────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(epochs, history['mae'], color='#e67e22', linewidth=2)
ax.set_title('MAE over Training'); ax.set_xlabel('Epoch'); ax.set_ylabel('MAE')
ax.grid(True, alpha=0.3)

plt.tight_layout()
loss_plot_path = os.path.join(DRIVE_DIR, 'training_curves.png')
plt.savefig(loss_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Loss curves saved → {loss_plot_path}")

## 8 · Confusion Matrix (Demand-Level Classification)

In [ ]:
# Bin demand into 3 classes: Low / Medium / High
# Uses percentiles so classes are balanced regardless of data scale.

trained_model.eval()
loader_eval = DataLoader(dataset, batch_size=128, shuffle=False)

all_true, all_pred_vals = [], []
all_mu_vecs = []

with torch.no_grad():
    for batch_x, batch_cond in loader_eval:
        x, cond = batch_x.to(DEVICE), batch_cond.to(DEVICE)
        recon, mu, logvar = trained_model(x, cond)
        # Only demand channels (first NUM_NODES channels)
        true_demand = x[:, :CFG.NUM_NODES, :].cpu().numpy()      # [B, 32, 24]
        pred_demand = recon[:, :CFG.NUM_NODES, :].cpu().numpy()
        all_true.append(true_demand.flatten())
        all_pred_vals.append(pred_demand.flatten())
        all_mu_vecs.append(mu.cpu().numpy())

true_flat = np.concatenate(all_true)
pred_flat = np.concatenate(all_pred_vals)

# Define class boundaries from true distribution
q33 = np.percentile(true_flat, 33)
q67 = np.percentile(true_flat, 67)
labels_arr = ['Low', 'Medium', 'High']

def to_class(arr):
    out = np.zeros(len(arr), dtype=int)
    out[arr > q33] = 1
    out[arr > q67] = 2
    return out

y_true_cls = to_class(true_flat)
y_pred_cls = to_class(pred_flat)

cm = confusion_matrix(y_true_cls, y_pred_cls)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100   # row-wise %

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EVolvAI — Demand-Level Confusion Matrix', fontsize=14, fontweight='bold')

cmap = LinearSegmentedColormap.from_list('ev_cmap', ['#eaf2ff', '#2980b9'])

for ax, data, fmt, title in zip(
        axes,
        [cm, cm_pct],
        ['d', '.1f'],
        ['Counts', 'Row-Normalised (%)']):
    im = ax.imshow(data, interpolation='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    thresh = data.max() / 2
    for i in range(3):
        for j in range(3):
            txt = f"{data[i,j]:{fmt}}"
            if fmt.endswith('f'): txt += '%'
            ax.text(j, i, txt, ha='center', va='center',
                    color='white' if data[i, j] > thresh else 'black', fontsize=12)
    ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
    ax.set_xticklabels(labels_arr); ax.set_yticklabels(labels_arr)
    ax.set_xlabel('Predicted Demand Class'); ax.set_ylabel('True Demand Class')
    ax.set_title(title)

plt.tight_layout()
cm_plot_path = os.path.join(DRIVE_DIR, 'confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Confusion matrix saved → {cm_plot_path}")

# ── Classification report ────────────────────────────────────────────────────
from sklearn.metrics import classification_report
print("\n📋 Classification Report:")
print(classification_report(y_true_cls, y_pred_cls, target_names=labels_arr))

## 9 · Key Characteristics Summary

In [ ]:
# ── Summary metrics ───────────────────────────────────────────────────────────
final_r2  = history['r2'][-1]
final_mae = history['mae'][-1]
final_loss = history['total_loss'][-1]

zero_pct = np.mean(np.abs(pred_flat) < 1e-4) * 100

print("=" * 55)
print("  EVolvAI — Key Training Characteristics")
print("=" * 55)
print(f"  Total parameters     : {total_params:,}")
print(f"  Training device      : {DEVICE}")
print(f"  Dataset samples      : {len(dataset)}")
print(f"  Final total loss     : {final_loss:.5f}")
print(f"  Final R²             : {final_r2:+.4f}")
print(f"  Best R²              : {max(history['r2']):+.4f}  (Epoch {epochs[history['r2'].index(max(history['r2']))]})") 
print(f"  Final MAE            : {final_mae:.4f}")
print(f"  Zero-output %        : {zero_pct:.2f}%  (target < 5%)")
print("=" * 55)

# ── Latent Space PCA ─────────────────────────────────────────────────────────
mu_all = np.concatenate(all_mu_vecs, axis=0)   # [N_samples, LATENT_DIM]
pca = PCA(n_components=2)
z_pca = pca.fit_transform(mu_all)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(z_pca[:, 0], z_pca[:, 1],
                c=y_true_cls[:len(z_pca)], cmap='coolwarm', alpha=0.6, s=15)
plt.colorbar(sc, ax=ax, ticks=[0, 1, 2], label='Demand class')
ax.set_title('Latent Space PCA (µ vectors, coloured by demand class)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
pca_plot_path = os.path.join(DRIVE_DIR, 'latent_pca.png')
plt.savefig(pca_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Latent PCA saved → {pca_plot_path}")
print(f"   PCA variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  PC2={pca.explained_variance_ratio_[1]*100:.1f}%")